# Download Data

Next, let's get the actual relevant gene counts. 

- h5ads can be found [here](https://lamin.ai/laminlabs/arc-virtual-cell-atlas/artifacts?filter[and][0][or][0][branch.name][eq]=Main&filter[and][1][or][0][is_latest][eq]=true&filter[and][2][or][0][projects.name][eq]=Tahoe-100M&filter[and][3][or][0][otype][eq]=AnnData)
- processing adapted from [here](https://theislab.github.io/vevo_Tahoe_100m_analysis/vevo_100m_pca.html):

In [1]:
from multiprocessing import Pool

import numpy as np
import scanpy as sc
import anndata as ad
import h5py

import pandas as pd
from tqdm import tqdm
import time

import os

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/nobackup/

In [2]:
n_cores = 80
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)
# os.environ["DASK_DISTRIBUTED__WORKER__PRELOAD"] = ""

# SPARSE_CHUNK_SIZE = 100_000
# cluster = LocalCluster(n_workers=n_cores)
# client = Client(cluster)
# mod = sc

In [3]:
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis/'
author = 'Tahoe100M'

In [4]:
url_name = 'https://storage.googleapis.com/arc-ctc-tahoe100/2025-02-25/h5ad/plate{}_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad'


obs_final = pd.read_csv(os.path.join(data_path, 'processed', '_filtered_cell_metadata.csv'), 
                            index_col = 0)
plate_iterable = sorted([int(plate_no.split('plate')[1]) for plate_no in sorted(obs_final.plate.unique())])

Download raw data:

In [ ]:
# # download all at once (will be needed to write final adata to memory)
# # ensure 380GB of memory is available
# # cmd = 'seq {} {}'.format(list(plate_iterable)[0], list(plate_iterable)[-1])
# cmd = 'echo ' + ' '.join(plate_iterable)
# cmd += ' | xargs -n 1 -P {} '.format(min(7, len(plate_iterable)))
# cmd += '-I {} wget -O '
# cmd += os.path.join(data_path, 'raw', os.path.basename(url_name))
# cmd += ' ' + url_name
# os.system(cmd)

Find the barcodes that we have retained:

In [5]:
plate_iterable

[3, 6, 9, 12, 13, 14]

In [34]:
adata_list = []
for plate_no in plate_iterable:
#     adata = get_tahoe_adata(plate_no)
    print('Start processing plate {}'.format(plate_no))

    url_name_iter = url_name.format(plate_no)
    fn = os.path.basename(url_name_iter)
    file_path = os.path.join(data_path, 'raw', fn)
    
    # download the data
    if not os.path.isfile(file_path):
        cmd = 'wget -O ' + file_path +  ' ' + url_name_iter
        os.system(cmd)
    
    
    # get the relevant barcode subset for this plate
    obs_final = pd.read_csv(os.path.join(data_path, 'processed', '_filtered_cell_metadata.csv'), 
                            index_col = 0)
    keep_barcodes = obs_final[obs_final.plate == 'plate{}'.format(plate_no)].index.tolist()

#     with h5py.File(file_path, "r") as f:
#         adata = ad.AnnData(
#             obs=ad.io.read_elem(f["obs"]),
#             var=ad.io.read_elem(f["var"]),
#         )
#         adata.X = ad.experimental.read_elem_as_dask(
#             f["X"], chunks=(SPARSE_CHUNK_SIZE, adata.shape[1])
#         )

    adata = ad.read_h5ad(file_path)
        
    barcodes = adata.obs.index.tolist()

    def check_membership(b):
        return b in keep_barcodes

    with Pool(processes=n_cores) as pool:
        mask = list(tqdm(pool.imap(check_membership, barcodes, chunksize=1000), total=len(barcodes)))

    adata = adata[mask, :].copy()

    sc.pp.normalize_total(adata, target_sum = 1e6)
    sc.pp.log1p(adata)
    # sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    
#     adata.write_h5ad(os.path.join(data_path, 'interim', 
#                                   author + '_plate{}'.format(plate_no) + '_filtered_normalized_counts.h5ad'))
    
    adata_list.append(adata)
    
    # remove downloaded data
    cmd = 'rm ' + file_path
    os.system(cmd)

Convert to memory and write:

In [ ]:
# adata_list = [adata.to_memory(copy = False) for adata in tqdm(adata_list)]
print('Merge adatas')
adata_combined = ad.concat(adata_list, join = 'outer')
print('Write file')
adata_combined.write_h5ad(os.path.join(data_path, 'interim', author + '_filtered_normalized_counts.h5ad'))


Remove large raw files:

In [ ]:
# cmd = 'rm ' + os.path.join(data_path, 'raw', 
#             os.path.basename(url_name).replace("{}", "*"))
# os.system(cmd)